In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import json
import pickle
from pyvis.network import Network

In [2]:
chunk_size = 200
octis_folder = f"./octis_{chunk_size}/"
optuna_folder = f"./optuna_{chunk_size}/"

In [3]:
df_metadata = pd.read_csv("./data/ChiLit_metadata.csv", encoding="utf-8")
df_authors = pd.read_csv("./data/ChiLit_Authors.csv", encoding="utf-8")

In [4]:
df_chilit = pd.read_csv(f"./data/ChiLit_Chunks_{chunk_size}.csv")
df_chilit = df_chilit.fillna("")

In [5]:
# ProdLDA model optimized by OCTIS
final_model = pickle.load(open(octis_folder + "Octis_ProdLDA_output.pkl", "rb"))

In [6]:
df_topic_word_probs = pd.read_csv(octis_folder + 'Octis_ProdLDA_topic_word_probs.csv', encoding='utf-8')

In [7]:
with open(octis_folder + "OCTIS_ProdLDA_Topic_Labels.json", 'r') as file:
  labels = json.load(file)

In [8]:
topic_labels = [value['primary_label'] for value in labels.values()]

In [9]:
# Add document original information (book, chapter)
n_topics = len(final_model['topics'])
df_topics = pd.DataFrame(final_model['topic-document-matrix'].T, columns=topic_labels)
df_topics['book_id'] = df_chilit['book_id'].to_list()
df_topics['chapter_num'] = df_chilit['chapter_num'].to_list()

In [10]:
# Aggreagate topics by book
agg_df = df_topics.drop(['chapter_num'], axis=1).groupby("book_id").mean()

In [11]:
colormap = plt.colormaps['tab20'].colors  # can be 'hsv', 'tab20', 'nipy_spectral', etc.
color_sequence = [mcolors.to_hex(colormap[i]) for i in range(n_topics)]

In [12]:
def generate_interactive_barchart(df, output_file="topic_relevance_barchart.html", title="Topic Relevance Explorer", color_sequence=px.colors.qualitative.Set3):
    """
    Generate a standalone HTML file with an interactive stacked bar chart.
    
    Parameters:
    - df: DataFrame with books as index and topics as columns
    - output_file: Name of the output HTML file
    - title: Title for the chart
    """
    
    # Reset index to get book_id as a column
    df_reset = df.reset_index()
    topics = df.columns.tolist()
    
    # Convert DataFrame to JSON for JavaScript
    df_json = df_reset.to_json(orient='records')
    
    # Create initial chart with all topics
    df_long = df_reset.melt(id_vars="book_id", 
                           value_vars=topics, 
                           var_name="Topic", 
                           value_name="Relevance")
    
    order = df_long.groupby("book_id")["Relevance"].sum().sort_values(ascending=False).index
    
    initial_fig = px.bar(
        df_long,
        x="Relevance",
        y="book_id",
        color="Topic",
        orientation="h",
        title=f"Books sorted by: {', '.join(topics)}",
        category_orders={"book_id": order},
        color_discrete_sequence=color_sequence#px.colors.qualitative.Set3
    )
    
    initial_fig.update_layout(
        barmode="stack", 
        yaxis_title="Book ID", 
        xaxis_title="Relevance", 
        height=1400,
        margin=dict(l=50, r=50, t=100, b=50)
    )
    
    # Generate the HTML content
    html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{title}</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            background-color: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h2 {{
            color: #333;
            text-align: center;
            margin-bottom: 30px;
        }}
        .controls {{
            margin-bottom: 20px;
            padding: 15px;
            background-color: #f9f9f9;
            border-radius: 5px;
            border: 1px solid #ddd;
        }}
        .topic-selector {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            align-items: center;
        }}
        .topic-selector label {{
            font-weight: bold;
            margin-right: 15px;
        }}
        .topic-checkbox {{
            margin: 5px 15px 5px 5px;
        }}
        .topic-checkbox input {{
            margin-right: 5px;
        }}
        #plotly-div {{
            width: 100%;
            height: 1400px;
        }}
        .select-all-btn {{
            background-color: #007bff;
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 4px;
            cursor: pointer;
            margin-left: 10px;
        }}
        .select-all-btn:hover {{
            background-color: #0056b3;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h2>{title}</h2>
        
        <div class="controls">
            <div class="topic-selector">
                <label>Select Topics:</label>
                <button class="select-all-btn" onclick="selectAllTopics()">Select All</button>
                <button class="select-all-btn" onclick="clearAllTopics()">Clear All</button>
            </div>
            <div id="topic-checkboxes">
                {' '.join([f'<div class="topic-checkbox"><input type="checkbox" id="topic_{i}" value="{topic}" onchange="updateChart()" checked><label for="topic_{i}">{topic}</label></div>' for i, topic in enumerate(topics)])}
            </div>
        </div>
        
        <div id="plotly-div"></div>
    </div>

    <script>
        // Data
        const rawData = {df_json};
        const topics = {json.dumps(topics)};
        const colors = {json.dumps(color_sequence)};       
        
        
        // Initialize with all topics selected
        updateChart();
        
        function getSelectedTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]:checked');
            return Array.from(checkboxes).map(cb => cb.value);
        }}
        
        function selectAllTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]');
            checkboxes.forEach(cb => cb.checked = true);
            updateChart();
        }}
        
        function clearAllTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]');
            checkboxes.forEach(cb => cb.checked = false);
            updateChart();
        }}
        
        function updateChart() {{
            const selectedTopics = getSelectedTopics();
            
            if (selectedTopics.length === 0) {{
                const layout = {{
                    title: "Please select at least one topic",
                    xaxis: {{title: "Relevance"}},
                    yaxis: {{title: "Book ID"}},
                    height: 400
                }};
                Plotly.newPlot('plotly-div', [], layout);
                return;
            }}
            
            // Transform data to long format
            const longData = [];
            rawData.forEach(row => {{
                selectedTopics.forEach(topic => {{
                    if (row[topic] !== undefined && row[topic] !== null) {{
                        longData.push({{
                            book_id: row.book_id,
                            topic: topic,
                            relevance: row[topic]
                        }});
                    }}
                }});
            }});
            
            // Calculate total relevance per book for sorting
            const bookTotals = {{}};
            longData.forEach(item => {{
                if (!bookTotals[item.book_id]) {{
                    bookTotals[item.book_id] = 0;
                }}
                bookTotals[item.book_id] += item.relevance;
            }});
            
            // Sort books by total relevance (descending)
            const sortedBooks = Object.keys(bookTotals)
                .sort((a, b) => bookTotals[b] - bookTotals[a]);
            
            // Create traces for each topic
            const traces = [];
            selectedTopics.forEach((topic, index) => {{
                const topicData = longData.filter(item => item.topic === topic);
                const x = [];
                const y = [];
                const text = [];
                
                sortedBooks.forEach(bookId => {{
                    const item = topicData.find(d => d.book_id === bookId);
                    const relevance = item ? item.relevance : 0;
                    x.push(relevance);
                    y.push(bookId);
                    text.push(`${{topic}}: ${{relevance.toFixed(2)}}`);
                }});
                
                traces.push({{
                    x: x,
                    y: y,
                    text: text,
                    name: topic,
                    type: 'bar',
                    orientation: 'h',
                    marker: {{
                        color: colors[index % colors.length]
                    }},
                    hovertemplate: '<b>%{{y}}</b><br>%{{text}}<br><extra></extra>'
                }});
            }});
            
            const layout = {{
                title: `Books sorted by: ${{selectedTopics.join(', ')}}`,
                xaxis: {{
                    title: 'Relevance'
                }},
                yaxis: {{
                    title: 'Book ID',
                    categoryorder: 'array',
                    categoryarray: sortedBooks.reverse() // Reverse for proper ordering
                }},
                barmode: 'stack',
                height: 1400,
                margin: {{l: 150, r: 50, t: 100, b: 50}},
                hovermode: 'closest'
            }};
            
            Plotly.newPlot('plotly-div', traces, layout);
        }}
    </script>
</body>
</html>
"""
    
    # Write to file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return output_file



generate_interactive_barchart(agg_df,output_file="./visualizations/topic_relevance_barchart.html")

'./visualizations/topic_relevance_barchart.html'

In [13]:
def generate_interactive_barchart(df, output_file="topic_relevance_barchart.html", title="Topic Relevance Explorer", color_sequence=px.colors.qualitative.Set3):
    """
    Generate a standalone HTML file with an interactive stacked bar chart.

    Parameters:
    - df: DataFrame with books as index and topics as columns
    - output_file: Name of the output HTML file
    - title: Title for the chart
    """

    import json
    import plotly.express as px

    # Reset index to get book_id as a column
    df_reset = df.reset_index()
    topics = df.columns.tolist()

    # Convert DataFrame to JSON for JavaScript
    df_json = df_reset.to_json(orient='records')

    # Prepare initial figure (not embedded, just for layout reference)
    df_long = df_reset.melt(id_vars="book_id",
                           value_vars=topics,
                           var_name="Topic",
                           value_name="Relevance")

    order = df_long.groupby("book_id")["Relevance"].sum().sort_values(ascending=False).index

    initial_fig = px.bar(
        df_long,
        x="Relevance",
        y="book_id",
        color="Topic",
        orientation="h",
        title=f"Books sorted by: {', '.join(topics)}",
        category_orders={"book_id": order},
    )

    initial_fig.update_layout(
        barmode="stack",
        yaxis_title="Book ID",
        xaxis_title="Relevance",
        height=1400,
        margin=dict(l=50, r=50, t=100, b=50)
    )

    # Generate the HTML content
    html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{title}</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            background-color: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h2 {{
            color: #333;
            text-align: center;
            margin-bottom: 30px;
        }}
        /* Flexbox layout for controls + chart */
        .main-content {{
            display: flex;
            gap: 20px;
            align-items: flex-start;
        }}
        .controls {{
            flex: 0 0 250px; /* sidebar width */
            padding: 15px;
            background-color: #f9f9f9;
            border-radius: 5px;
            border: 1px solid #ddd;
            max-height: 1400px;
            overflow-y: auto;
        }}
        #plotly-div {{
            flex: 1;
            height: 1400px;
        }}
        .topic-selector {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            align-items: center;
            margin-bottom: 10px;
        }}
        .topic-checkbox {{
            margin: 5px 0;
        }}
        .topic-checkbox input {{
            margin-right: 5px;
        }}
        .select-all-btn {{
            background-color: #007bff;
            color: white;
            border: none;
            padding: 6px 12px;
            border-radius: 4px;
            cursor: pointer;
            margin-right: 5px;
        }}
        .select-all-btn:hover {{
            background-color: #0056b3;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h2>{title}</h2>

        <div class="main-content">
            <!-- Sidebar controls -->
            <div class="controls">
                <div class="topic-selector">
                    <label>Select Topics:</label>
                </div>
                <div style="margin-bottom:10px;">
                    <button class="select-all-btn" onclick="selectAllTopics()">Select All</button>
                    <button class="select-all-btn" onclick="clearAllTopics()">Clear All</button>
                </div>
                <div id="topic-checkboxes">
                    {' '.join([f'<div class="topic-checkbox"><input type="checkbox" id="topic_{i}" value="{topic}" onchange="updateChart()" checked><label for="topic_{i}">{topic}</label></div>' for i, topic in enumerate(topics)])}
                </div>
            </div>

            <!-- Chart -->
            <div id="plotly-div"></div>
        </div>
    </div>

    <script>
        // Data
        const rawData = {df_json};
        const topics = {json.dumps(topics)};
        const colors = {json.dumps(color_sequence)};       

        // Initialize with all topics selected
        updateChart();

        function getSelectedTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]:checked');
            return Array.from(checkboxes).map(cb => cb.value);
        }}

        function selectAllTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]');
            checkboxes.forEach(cb => cb.checked = true);
            updateChart();
        }}

        function clearAllTopics() {{
            const checkboxes = document.querySelectorAll('input[type="checkbox"]');
            checkboxes.forEach(cb => cb.checked = false);
            updateChart();
        }}

        function updateChart() {{
            const selectedTopics = getSelectedTopics();

            if (selectedTopics.length === 0) {{
                const layout = {{
                    title: "Please select at least one topic",
                    xaxis: {{title: "Relevance"}},
                    yaxis: {{title: "Book ID"}},
                    height: 400
                }};
                Plotly.newPlot('plotly-div', [], layout);
                return;
            }}

            // Transform data to long format
            const longData = [];
            rawData.forEach(row => {{
                selectedTopics.forEach(topic => {{
                    if (row[topic] !== undefined && row[topic] !== null) {{
                        longData.push({{
                            book_id: row.book_id,
                            topic: topic,
                            relevance: row[topic]
                        }});
                    }}
                }});
            }});

            // Calculate total relevance per book for sorting
            const bookTotals = {{}};
            longData.forEach(item => {{
                if (!bookTotals[item.book_id]) {{
                    bookTotals[item.book_id] = 0;
                }}
                bookTotals[item.book_id] += item.relevance;
            }});

            // Sort books by total relevance (descending)
            const sortedBooks = Object.keys(bookTotals)
                .sort((a, b) => bookTotals[b] - bookTotals[a]);

            // Create traces for each topic
            const traces = [];
            selectedTopics.forEach((topic, index) => {{
                const topicData = longData.filter(item => item.topic === topic);
                const x = [];
                const y = [];
                const text = [];

                sortedBooks.forEach(bookId => {{
                    const item = topicData.find(d => d.book_id === bookId);
                    const relevance = item ? item.relevance : 0;
                    x.push(relevance);
                    y.push(bookId);
                    text.push(`${{topic}}: ${{relevance.toFixed(2)}}`);
                }});

                traces.push({{
                    x: x,
                    y: y,
                    text: text,
                    name: topic,
                    type: 'bar',
                    orientation: 'h',
                    marker: {{
                        color: colors[index % colors.length]
                    }},
                    hovertemplate: '<b>%{{y}}</b><br>%{{text}}<br><extra></extra>'
                }});
            }});

            const layout = {{
                title: `Books sorted by: ${{selectedTopics.join(', ')}}`,
                xaxis: {{
                    title: 'Relevance'
                }},
                yaxis: {{
                    title: 'Book ID',
                    categoryorder: 'array',
                    categoryarray: sortedBooks.reverse()
                }},
                barmode: 'stack',
                height: 1400,
                margin: {{l: 150, r: 50, t: 100, b: 50}},
                hovermode: 'closest'
            }};

            Plotly.newPlot('plotly-div', traces, layout);
        }}
    </script>
</body>
</html>
"""

    # Write to file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)

    return output_file

generate_interactive_barchart(agg_df,output_file="./visualizations/topic_relevance_barchart.html",color_sequence=color_sequence)

'./visualizations/topic_relevance_barchart.html'

In [14]:
import plotly.graph_objects as go
import pandas as pd

def generate_interactive_heatmap(agg_df, title="Topic Relevance Heatmap", output_file="topic_relevance_heatmap.html", row_height=30):
    """
    Create an advanced interactive heatmap with individual book selection.
    Users can check/uncheck individual books to include/exclude from visualization.
    Row height is fixed (in pixels) per row, hidden rows disappear without changing row scaling.
    """
    
    # Prepare data
    all_books = list(agg_df.index)
    
    # Create figure
    fig = go.Figure()
    
    # Add initial trace with all books
    fig.add_trace(
        go.Heatmap(
            z=agg_df.values,
            x=agg_df.columns,
            y=agg_df.index,
            colorscale='YlGnBu',
            hoverongaps=False,
            hovertemplate='Document: %{y}<br>Topic: %{x}<br>Relevance: %{z:.4f}<extra></extra>',
            colorbar=dict(title="Relevance Score"),
            name="Selected Books"
        )
    )
    
    # JavaScript code with dynamic relayout to fix row height
    custom_js = f"""
    <script>
    let selectedBooks = {all_books};
    let allData = {agg_df.to_dict('list')};
    let allBooks = {all_books};
    let columns = {list(agg_df.columns)};
    const rowHeight = {row_height};

    function updateHeatmap() {{
        let filteredData = [];
        let filteredLabels = [];
        
        selectedBooks.forEach(book => {{
            let bookIndex = allBooks.indexOf(book);
            filteredLabels.push(book);
            let row = [];
            columns.forEach(col => {{
                row.push(allData[col][bookIndex]);
            }});
            filteredData.push(row);
        }});

        Plotly.restyle('heatmap-div', {{
            'z': [filteredData],
            'y': [filteredLabels]
        }}, [0]);

        // Update figure height so each row has fixed pixel height
        let newHeight = rowHeight * filteredLabels.length + 300;
        Plotly.relayout('heatmap-div', {{
            height: newHeight
        }});
    }}
    
    function toggleBook(book) {{
        let index = selectedBooks.indexOf(book);
        if (index > -1) {{
            selectedBooks.splice(index, 1);
        }} else {{
            selectedBooks.push(book);
        }}
        updateHeatmap();
        updateButtonLabels();
    }}
    
    function selectAll() {{
        selectedBooks = [...allBooks];
        updateHeatmap();
        updateButtonLabels();
    }}
    
    function deselectAll() {{
        selectedBooks = [];
        updateHeatmap();
        updateButtonLabels();
    }}
    
    function updateButtonLabels() {{
        allBooks.forEach(book => {{
            let button = document.getElementById('btn-' + book.replace(/[^a-zA-Z0-9]/g, ''));
            if (button) {{
                if (selectedBooks.includes(book)) {{
                    button.innerHTML = '✓ ' + book;
                    button.className = 'book-btn selected';
                }} else {{
                    button.innerHTML = '✗ ' + book;
                    button.className = 'book-btn unselected';
                }}
            }}
        }});
    }}
    </script>
    
    <style>
    .book-controls {{
        margin: 20px 0;
        padding: 15px;
        border: 1px solid #ddd;
        border-radius: 5px;
        background-color: #f9f9f9;
    }}
    
    .book-btn {{
        margin: 3px;
        padding: 5px 10px;
        border: 1px solid #ccc;
        border-radius: 3px;
        cursor: pointer;
        display: inline-block;
        font-size: 12px;
        transition: all 0.2s;
    }}
    
    .book-btn.selected {{
        background-color: #4CAF50;
        color: white;
        border-color: #45a049;
    }}
    
    .book-btn.unselected {{
        background-color: #adb5bd;
        color: white;
        border-color: #495057;
    }}
    
    .control-btn {{
        margin: 5px;
        padding: 8px 15px;
        border: 1px solid #2196F3;
        border-radius: 4px;
        background-color: #2196F3;
        color: white;
        cursor: pointer;
        font-weight: bold;
    }}
    
    .control-btn:hover {{
        background-color: #1976D2;
    }}
    </style>
    
    <div class="book-controls">
        <h3>Select Books to Display:</h3>
        <div>
            <span class="control-btn" onclick="selectAll()">Select All</span>
            <span class="control-btn" onclick="deselectAll()">Deselect All</span>
        </div>
        <br>
        <div id="book-buttons">
    """
    
    # Add individual book buttons to HTML
    for book in all_books:
        safe_id = ''.join(c for c in book if c.isalnum())
        custom_js += f'<span id="btn-{safe_id}" class="book-btn selected" onclick="toggleBook(\'{book}\')">{book}</span>\n'
    
    custom_js += """
        </div>
    </div>
    """
    
    # Initial layout with height proportional to number of books
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            font=dict(size=16)
        ),
        xaxis_title="Topics",
        yaxis_title="Books", 
        height=row_height * len(all_books) + 300,
        width=1200,
        font=dict(size=10),
        xaxis=dict(
            tickangle=45,
            side='bottom'
        ),
        yaxis=dict(
            tickangle=0,
            autorange='reversed'
        ),
        margin=dict(t=100, l=150, r=50, b=150)
    )
    
    # Save as HTML with custom controls
    html_string = fig.to_html(include_plotlyjs='cdn', div_id='heatmap-div')
    
    # Insert custom controls before the plot
    html_parts = html_string.split('<body>')
    if len(html_parts) == 2:
        html_string = html_parts[0] + '<body>' + custom_js + html_parts[1]
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_string)

    return fig

data = agg_df.T.to_dict(orient='list')
columns = agg_df.columns.tolist()

# Create DataFrame
df = pd.DataFrame(data, index=columns).T
fig = generate_interactive_heatmap(df, output_file="./visualizations/topic_relevance_heatmap.html")

In [15]:


book_topic_df = agg_df  

books = list(book_topic_df.index)
topics = list(book_topic_df.columns)

# Parameters
n_top = 2
B = nx.Graph()

# Add nodes
B.add_nodes_from(book_topic_df.index, bipartite="books")
B.add_nodes_from(book_topic_df.columns, bipartite="topics")

# Add edges (book → top n topics)
for book, row in book_topic_df.iterrows():
    top_topics = row.nlargest(n_top).index
    for topic in top_topics:
        B.add_edge(book, topic, weight=row[topic])

# Create PyVis network
net = Network(height="800px", width="100%", notebook=False, bgcolor="#ffffff", font_color="black")
net.from_nx(B)

# Customize node styles
for node in net.nodes:
    if node['id'] in books:
        node['color'] = {
            "background": "lightblue",
            "border": "#2b7ce9",
            "highlight": {"background": "lightblue", "border": "#2b7ce9"},
            "hover": {"background": "lightblue", "border": "#2b7ce9"}
        }
        node['size'] = 15 + 3 * B.degree(node['id'])
        node['title'] = f"Book: {node['id']}"
        node['group'] = "books"
        node['font'] = {"size": 20}
    else:
        node['color'] = {
            "background": "lightgreen",
            "border": "#2b7ce9",
            "highlight": {"background": "lightgreen", "border": "#2b7ce9"},
            "hover": {"background": "lightgreen", "border": "#2b7ce9"}
        }
        node['size'] = 20 + 200 * book_topic_df[node['id']].mean()
        node['title'] = f"Topic: {node['id']}"
        node['group'] = "topics"
        node['font'] = {"size": 22}

net.toggle_physics(True)

# Save PyVis HTML
net.save_graph("./visualizations/graph_base.html")

# Extract <head> and <body> from PyVis HTML
head_html, body_html = [], []
inside_head, inside_body = False, False

with open("./visualizations/graph_base.html", "r", encoding="utf-8") as f:
    for line in f:
        if "<head>" in line:
            inside_head = True
            continue
        if "</head>" in line:
            inside_head = False
        if inside_head:
            head_html.append(line)

        if "<body>" in line:
            inside_body = True
            continue
        if "</body>" in line:
            inside_body = False
        if inside_body:
            body_html.append(line)

head_html = "".join(head_html)
graph_html = "".join(body_html)

# Sidebar checkboxes
checkbox_html = "\n".join(
    [f'<label><input type="checkbox" class="topic-checkbox" checked onchange="toggleTopic(\'{topic}\')"> {topic}</label><br>'
     for topic in topics]
)

# Final HTML with JS
final_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Interactive Book–Topic Graph</title>
    {head_html}
    <style>
        body {{
            font-family: Arial, sans-serif;
        }}
        #controls {{
            position: fixed;
            top: 10px;
            left: 10px;
            background: #f9f9f9;
            padding: 10px;
            border: 1px solid #ccc;
            border-radius: 8px;
            max-height: 90vh;
            overflow-y: auto;
            z-index: 9999;
        }}
        #controls button {{
            margin: 5px 0;
            width: 100%;
            padding: 5px;
            border-radius: 5px;
            cursor: pointer;
        }}
        #mynetwork {{
            margin-left: 220px;
        }}
    </style>
</head>
<body>
    <div id="controls">
        <h3>Topics</h3>
        <button onclick="hideAll()">Hide All</button>
        <button onclick="showAll()">Show All</button>
        <hr>
        {checkbox_html}
    </div>

    {graph_html}

    <script>
        // Save original colors for all nodes
        var originalColors = {{}};
        network.body.data.nodes.get().forEach(function(n) {{
            originalColors[n.id] = n.color;
        }});

        function updateBooks() {{
            var nodes = network.body.data.nodes.get();
            var topicVisible = {{}};
            nodes.forEach(function(n) {{
                if (n.group === "topics") {{
                    topicVisible[n.id] = !n.hidden;
                }}
            }});

            nodes.forEach(function(n) {{
                if (n.group === "books") {{
                    var connected = network.getConnectedNodes(n.id);
                    var visibleTopics = connected.filter(c => topicVisible[c]);
                    if (visibleTopics.length === 0) {{
                        network.body.data.nodes.update({{id: n.id, hidden: true}});
                    }} else {{
                        network.body.data.nodes.update({{
                            id: n.id,
                            hidden: false,
                            color: originalColors[n.id]
                        }});
                    }}
                }}
            }});
        }}

        function toggleTopic(topicId) {{
            var node = network.body.data.nodes.get(topicId);
            if (!node) return;
            if (node.hidden) {{
                network.body.data.nodes.update({{
                    id: topicId,
                    hidden: false,
                    color: originalColors[topicId]
                }});
            }} else {{
                network.body.data.nodes.update({{id: topicId, hidden: true}});
            }}
            updateBooks();
        }}

        function hideAll() {{
            var checkboxes = document.querySelectorAll('.topic-checkbox');
            checkboxes.forEach(cb => cb.checked = false);
            var nodes = network.body.data.nodes.get();
            nodes.forEach(function(n) {{
                if (n.group === "topics") {{
                    network.body.data.nodes.update({{id: n.id, hidden: true}});
                }}
            }});
            updateBooks();
        }}

        function showAll() {{
            var checkboxes = document.querySelectorAll('.topic-checkbox');
            checkboxes.forEach(cb => cb.checked = true);
            var nodes = network.body.data.nodes.get();
            nodes.forEach(function(n) {{
                if (n.group === "topics") {{
                    network.body.data.nodes.update({{
                        id: n.id,
                        hidden: false,
                        color: originalColors[n.id]
                    }});
                }}
            }});
            updateBooks();
        }}

        // Run once on load
        updateBooks();
    </script>
</body>
</html>
"""

with open("./visualizations/book_topic_network.html", "w", encoding="utf-8") as f:
    f.write(final_html)

